# WallStreet Data Visualization
This notebook provides visualizations for the financial data stored in the WallStreet database.
It queries the Postgres database directly to fetch stock prices, profit & loss statements, balance sheets, and cash flows to visualize historical trends.

In [ ]:
import psycopg2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os

# Set seaborn style for better aesthetics
sns.set_theme(style="darkgrid")

# Load environment variables to get the database url
load_dotenv('../.env')
db_url = os.getenv('DB_URL')

if not db_url:
    print("Please set DB_URL in the .env file")
else:
    print("Database URL found.")
    # Connect to the WallStreet database
    conn = psycopg2.connect(db_url)

## 1. Stock Prices Over Time
Visualize the closing stock prices of all companies over time.

In [ ]:
try:
    prices_query = """
        SELECT c.symbol, p.date, p.close_price 
        FROM prices p
        JOIN companies c ON p.company_id = c.id
        ORDER BY p.date ASC
    """
    
    df_prices = pd.read_sql(prices_query, conn)
    df_prices['date'] = pd.to_datetime(df_prices['date'])
    
    plt.figure(figsize=(14, 7))
    sns.lineplot(data=df_prices, x='date', y='close_price', hue='symbol', linewidth=2)
    
    plt.title('Historical Stock Prices', fontsize=16)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Closing Price (INR)', fontsize=12)
    plt.legend(title='Company', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Error fetching/plotting prices: {e}")

## 2. Revenue and Profitability (Profit & Loss)
Analyze the growth in sales and net profit over the years for a specific company (e.g., RELIANCE).

In [ ]:
try:
    # Query P&L data for a few top companies to compare Sales Growth
    pl_query = """
        SELECT c.symbol, EXTRACT(YEAR FROM pl.year)::int as year, pl.sales, pl.net_profit
        FROM profit_loss pl
        JOIN companies c ON pl.company_id = c.id
        ORDER BY pl.year ASC
    """
    
    df_pl = pd.read_sql(pl_query, conn)
    # Convert values from absolute to Crores (Dividing by 10 million since dummy data was scaled up)
    df_pl['sales_cr'] = df_pl['sales'] / 10000000
    df_pl['net_profit_cr'] = df_pl['net_profit'] / 10000000
    
    plt.figure(figsize=(14, 7))
    sns.barplot(data=df_pl[df_pl['symbol'] == 'RELIANCE'], x='year', y='sales_cr', color='#3498DB', alpha=1, label='Sales')
    sns.barplot(data=df_pl[df_pl['symbol'] == 'RELIANCE'], x='year', y='net_profit_cr', color='#27AE60', alpha=1, label='Net Profit')
    
    plt.title('Reliance Industries: Sales vs Net Profit (in Crores)', fontsize=16)
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Amount (Cr)', fontsize=12)
    plt.legend()
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Error fetching/plotting P&L: {e}")

## 3. Financial Health (Debt to Equity Trend)
Compare the Debt-to-Equity Ratio for different companies across the years.

In [ ]:
try:
    ratio_query = """
        SELECT c.symbol, EXTRACT(YEAR FROM r.year)::int as year, r.debt_equity
        FROM ratios r
        JOIN companies c ON r.company_id = c.id
        ORDER BY r.year ASC
    """
    
    df_ratios = pd.read_sql(ratio_query, conn)
    
    plt.figure(figsize=(14, 7))
    sns.lineplot(data=df_ratios, x='year', y='debt_equity', hue='symbol', marker='o', linewidth=2)
    
    plt.title('Debt-to-Equity Ratio Trend', fontsize=16)
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Debt/Equity', fontsize=12)
    # Add a horizontal line at 1.0 (generally considered a healthy threshold)
    plt.axhline(1.0, color='red', linestyle='--', alpha=0.5, label='Threshold (1.0)')
    plt.legend(title='Company', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Error fetching/plotting ratios: {e}")